# Multi-Agent Systems with a Local Mistral Model

This is a local-LLM version of the Day 1 multi-agent notebook. It keeps the same workflow patterns, but swaps the cloud Gemini model for a local Mistral model served through an OpenAI-compatible endpoint such as Ollama or LM Studio.

The notebook is intended to run in the `temp` Python environment used by this workspace.

## 1. Local setup

Before running the notebook, start a local model server and make sure it is reachable at an OpenAI-compatible endpoint.

For Ollama, the usual setup is:

```bash
brew install ollama
ollama serve
ollama pull mistral

# after only need to run 
ollama serve
```

If you use a different local server, update `OLLAMA_BASE_URL` in the next cell.

In [3]:
import os
import json
from typing import AsyncGenerator, Any

import httpx
from openai import AsyncOpenAI

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.base_llm import BaseLlm
from google.adk.models.llm_response import LlmResponse
from google.adk.runners import InMemoryRunner
from google.genai import types

print("✅ Local notebook imports loaded.")

✅ Local notebook imports loaded.


In [4]:
MODEL_NAME = os.getenv("MISTRAL_MODEL", "mistral")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

def _api_root(base_url: str) -> str:
    return base_url[:-3] if base_url.endswith("/v1") else base_url

def _check_local_model_server() -> None:
    root_url = _api_root(OLLAMA_BASE_URL)
    try:
        response = httpx.get(f"{root_url}/api/tags", timeout=5.0)
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f"Could not reach a local model server at {root_url}. Start Ollama or set OLLAMA_BASE_URL to your local OpenAI-compatible endpoint."
        ) from exc

_check_local_model_server()
print(f"✅ Local model server is reachable. Using model: {MODEL_NAME}")

✅ Local model server is reachable. Using model: mistral


## 2. Local ADK model adapter

The class below adapts an OpenAI-compatible local model server to ADK's `BaseLlm` interface. It keeps the notebook compatible with the rest of ADK's agent and workflow APIs.

In [5]:
def _content_to_text(content: types.Content) -> str:
    parts = []
    for part in content.parts:
        if getattr(part, "text", None):
            parts.append(part.text)
    return "".join(parts)

class LocalMistralLLM(BaseLlm):
    model: str = MODEL_NAME
    base_url: str = OLLAMA_BASE_URL
    api_key: str = "ollama"

    async def generate_content_async(self, llm_request, stream: bool = False) -> AsyncGenerator[LlmResponse, None]:
        client = AsyncOpenAI(base_url=self.base_url, api_key=self.api_key)

        messages: list[dict[str, Any]] = []
        system_instruction = getattr(llm_request.config, "system_instruction", None)
        if system_instruction:
            messages.append({"role": "system", "content": str(system_instruction)})

        for content in llm_request.contents:
            text = _content_to_text(content)
            if not text.strip():
                continue
            role = content.role or "user"
            if role not in {"user", "assistant", "system"}:
                role = "user"
            messages.append({"role": role, "content": text})

        temperature = getattr(llm_request.config, "temperature", None)
        response = await client.chat.completions.create(
            model=self.model,
            messages=messages or [{"role": "user", "content": "Hello"}],
            temperature=temperature if temperature is not None else 0.2,
            stream=False,
        )

        text = response.choices[0].message.content or ""
        yield LlmResponse(
            content=types.Content(
                role="assistant",
                parts=[types.Part.from_text(text=text)],
            ),
            partial=False,
        )

local_model = LocalMistralLLM()
print("✅ LocalMistralLLM ready.")

✅ LocalMistralLLM ready.


## 3. Sequential workflow

This section recreates the blog pipeline pattern using the local model. The agents no longer rely on Google Search, so the prompts are written to work offline with the model's own knowledge.

In [8]:
outline_agent = Agent(
    name="OutlineAgent",
    model=local_model,
    instruction="Create a concise blog outline for the topic in the user prompt. Return 1 headline, 1 hook, 3 main sections, and a conclusion.",
    output_key="blog_outline",
)

writer_agent = Agent(
    name="WriterAgent",
    model=local_model,
    instruction="Using this outline: {blog_outline}. Write a clear 700-900 word blog post with an engaging tone.",
    output_key="blog_draft",
)

editor_agent = Agent(
    name="EditorAgent",
    model=local_model,
    instruction="Edit this draft for clarity, structure, and grammar: {blog_draft}. Return the improved final blog post only.",
    output_key="final_blog",
)

sequential_root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential workflow agents created.")

✅ Sequential workflow agents created.


/var/folders/y_/1pxsz__d753g2nqbbjh_gzh00000gn/T/ipykernel_63151/1683546479.py:22: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  sequential_root_agent = SequentialAgent(


In [9]:
runner = InMemoryRunner(agent=sequential_root_agent)
response = await runner.run_debug("Write a blog post about the benefits of AI for medical imaging")

Skipping missing token usage metadata for agent OutlineAgent and model mistral


OutlineAgent >  Title: Unleashing the Power of AI in Medical Imaging: Revolutionizing Diagnosis and Care

Hook:
Imagine a future where accurate, swift, and accessible medical diagnoses are just a click away. Welcome to the era of Artificial Intelligence (AI) in medical imaging!

Main Sections:
1. **Transformative Accuracy**: Exploring how AI enhances diagnostic precision by reducing human error, improving image analysis, and enabling early detection of diseases such as cancer.
2. **Efficiency Boost**: Discussing the time-saving potential of AI in medical imaging, from streamlining workflows to allowing radiologists to focus on complex cases.
3. **Accessibility and Equity**: Highlighting how AI can make quality healthcare more accessible by reducing the need for specialized expertise and enabling remote diagnosis in underserved areas.

Conclusion:
In conclusion, the integration of AI into medical imaging holds immense promise for improving patient outcomes, enhancing efficiency, and pro

Skipping missing token usage metadata for agent WriterAgent and model mistral


WriterAgent >  Title: Unleashing the Power of AI in Medical Imaging: Revolutionizing Diagnosis and Care

Welcome, dear readers, to a world where the future of healthcare is not just around the corner - it's already here! We're stepping into the era of Artificial Intelligence (AI) in medical imaging, a game-changer that promises to revolutionize diagnosis and care. So buckle up as we delve into this fascinating realm!

First off, let's talk about **Transformative Accuracy**. AI is like a superhero sidekick for our radiologists, reducing human error and improving image analysis with its unparalleled precision. Imagine an AI system that can spot the tiniest anomaly in an X-ray or MRI scan - it's no longer a distant dream but a reality. This enhanced diagnostic precision is particularly crucial in early detection of diseases such as cancer, where timely intervention can significantly improve patient outcomes.

Next up, we have the **Efficiency Boost**. AI is not just about accuracy; it's a

Skipping missing token usage metadata for agent EditorAgent and model mistral


EditorAgent >  Title: Revolutionizing Diagnosis and Care with Artificial Intelligence in Medical Imaging

Welcome, dear readers, to a world where the future of healthcare is no longer around the corner - it's here! We're stepping into the era of Artificial Intelligence (AI) in medical imaging, a game-changer that promises to revolutionize diagnosis and care. Buckle up as we delve into this fascinating realm!

First off, let's discuss **Transformative Accuracy**. AI serves as a superhero sidekick for our radiologists, reducing human error and improving image analysis with unparalleled precision. Imagine an AI system that can spot the tiniest anomaly in an X-ray or MRI scan - it's no longer a distant dream but a reality. This enhanced diagnostic precision is particularly crucial in early detection of diseases such as cancer, where timely intervention can significantly improve patient outcomes.

Next up, we have the **Efficiency Boost**. AI isn't just about accuracy; it's also about speed

## 4. Parallel workflow

This version keeps the same parallel orchestration idea, but each specialist uses the local Mistral model directly instead of a cloud-backed search tool.

In [10]:
tech_researcher = Agent(
    name="TechResearcher",
    model=local_model,
    instruction="Write a concise 120-word briefing on the latest AI and machine learning trends based on your general knowledge.",
    output_key="tech_research",
)

health_researcher = Agent(
    name="HealthResearcher",
    model=local_model,
    instruction="Write a concise 120-word briefing on current medical AI and health-tech trends based on your general knowledge.",
    output_key="health_research",
)

finance_researcher = Agent(
    name="FinanceResearcher",
    model=local_model,
    instruction="Write a concise 120-word briefing on current fintech and finance automation trends based on your general knowledge.",
    output_key="finance_research",
)

aggregator_agent = Agent(
    name="AggregatorAgent",
    model=local_model,
    instruction="""Combine these three briefings into one executive summary:

Technology Trends:
{tech_research}

Health Trends:
{health_research}

Finance Trends:
{finance_research}

Return around 200 words, with common themes and key takeaways.""",
    output_key="executive_summary",
)

parallel_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

parallel_root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_team, aggregator_agent],
)

print("✅ Parallel workflow agents created.")

✅ Parallel workflow agents created.


/var/folders/y_/1pxsz__d753g2nqbbjh_gzh00000gn/T/ipykernel_63151/3077076778.py:40: DeprecationWarning: ParallelAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  parallel_team = ParallelAgent(
/var/folders/y_/1pxsz__d753g2nqbbjh_gzh00000gn/T/ipykernel_63151/3077076778.py:45: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  parallel_root_agent = SequentialAgent(


In [11]:
runner = InMemoryRunner(agent=parallel_root_agent)
response = await runner.run_debug("Run a daily executive briefing on technology, health, and finance")

Skipping missing token usage metadata for agent FinanceResearcher and model mistral


FinanceResearcher >  Title: Fintech & Finance Automation Trends - Daily Executive Briefing (February 15, 2023)

Subject: Technology, Health, and Finance

FinanceResearcher Report:

1. Artificial Intelligence (AI): AI continues to revolutionize the finance sector with applications in fraud detection, investment management, and customer service. Notable advancements include the use of machine learning algorithms for credit scoring and robo-advisors for personalized investment advice.

2. Blockchain: Blockchain technology is transforming various aspects of finance, from cross-border payments to supply chain management. Major financial institutions are exploring its potential for increased security, transparency, and efficiency.

3. Open Banking: Regulatory initiatives such as PSD2 in Europe and Open Banking in the UK are driving the adoption of open APIs, enabling third-party providers to access banking data securely, fostering innovation and competition within the financial services indu

Skipping missing token usage metadata for agent TechResearcher and model mistral


TechResearcher >  TechResearcher Briefing - Day 123:

AI & Machine Learning Trends:

1. Quantum Computing Integration: Major tech companies are investing in quantum computing to enhance AI capabilities, promising faster processing speeds and improved machine learning algorithms.

2. Ethical AI Development: With growing concerns about privacy and bias, there's a push for more transparent and ethical AI development practices, including explainable AI (XAI) and fairness-aware machine learning.

3. Edge AI: To reduce latency and improve efficiency, edge AI is gaining traction. It allows AI processing to occur closer to the data source, reducing reliance on cloud infrastructure.

4. AI in Healthcare: AI is revolutionizing healthcare with applications ranging from disease diagnosis to personalized treatment plans. Notable advancements include AI-powered medical imaging and drug discovery.

5. Fintech AI: The financial sector continues to leverage AI for fraud detection, risk assessment, and 

Skipping missing token usage metadata for agent HealthResearcher and model mistral


HealthResearcher >  Title: Daily Executive Briefing - HealthTech & AI Trends (April 15, 2023)

Subject: HealthTech & AI Landscape

Key Points:
1. AI in Diagnostics: DeepMind's AlphaFold2 has revolutionized protein folding predictions, potentially accelerating drug discovery and treatment development.
2. Telemedicine Boom: The pandemic has accelerated telehealth adoption with platforms like Teladoc Health reporting a 45% increase in virtual visits year-over-year.
3. Mental Health AI: Advances in AI are being leveraged to improve mental health services, with companies like Mindstrong using machine learning to monitor and support patients with conditions such as depression and anxiety.
4. Precision Medicine: Genomic sequencing is becoming more accessible and affordable, enabling personalized treatment plans based on a patient's unique genetic makeup.
5. Financing HealthTech: Venture capital investments in digital health reached an all-time high of $16.8 billion in Q3 2022, indicating cont

Skipping missing token usage metadata for agent AggregatorAgent and model mistral


AggregatorAgent >  Executive Summary: Daily Technology, Health, and Finance Briefing - April 15, 2023

Artificial Intelligence (AI) is transforming various sectors, including finance, healthcare, and technology. In finance, AI is being utilized for fraud detection, investment management, credit scoring, and customer service through robo-advisors. Blockchain technology is also revolutionizing finance by enhancing security, transparency, and efficiency in cross-border payments and supply chain management.

Regulatory initiatives such as PSD2 in Europe and Open Banking in the UK are driving the adoption of open APIs, fostering innovation and competition within the financial services industry. Sustainable finance is gaining traction, with investors seeking to align their portfolios with environmental, social, and governance (ESG) principles. Fintech solutions are emerging to facilitate ESG investing and track the impact of investments on sustainability goals.

In healthcare, AI is revoluti

## 5. Loop workflow

This section demonstrates iterative refinement with a local model. The loop uses a writer and a critic, then repeats for a small number of iterations.

In [12]:
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=local_model,
    instruction="Write the first draft of a short story (100-150 words) based on the user's prompt. Return only the story text.",
    output_key="current_story",
)

critic_agent = Agent(
    name="CriticAgent",
    model=local_model,
    instruction="Review the story draft at {current_story}. If the story is strong, respond with APPROVED. Otherwise give 2-3 specific improvements.",
    output_key="critique",
)

refiner_agent = Agent(
    name="RefinerAgent",
    model=local_model,
    instruction="Use this critique to improve the story draft. Story draft: {current_story}. Critique: {critique}. Return the revised story only.",
    output_key="current_story",
)

story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,
)

loop_root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop workflow agents created.")

✅ Loop workflow agents created.


/var/folders/y_/1pxsz__d753g2nqbbjh_gzh00000gn/T/ipykernel_63151/3387464563.py:22: DeprecationWarning: LoopAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  story_refinement_loop = LoopAgent(
/var/folders/y_/1pxsz__d753g2nqbbjh_gzh00000gn/T/ipykernel_63151/3387464563.py:28: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  loop_root_agent = SequentialAgent(


In [13]:
runner = InMemoryRunner(agent=loop_root_agent)
response = await runner.run_debug("Write a short story about a lighthouse keeper who discovers a glowing map")

Skipping missing token usage metadata for agent InitialWriterAgent and model mistral


InitialWriterAgent >  In the solitary embrace of the rocky cliffs, Elijah tended to the beacon's flame. One stormy night, a peculiar sight caught his eye - a glowing map, washed ashore by the tempestuous sea.

The parchment pulsed with an ethereal light, its lines tracing an uncharted route towards a distant island. Intrigued, Elijah traced the map's path, venturing into the darkness, guided only by the lighthouse's beam and the glowing map's mysterious allure.

Days turned into nights as he traversed treacherous terrains, but the map remained his compass. On the eve of the tenth day, Elijah reached the island, where a hidden treasure awaited him - not gold or jewels, but knowledge, ancient and powerful.

With newfound wisdom, Elijah returned to his lighthouse, the glowing map now a cherished relic. His solitude was no longer lonely, for he carried within him secrets that could light the way for others lost at sea.


Skipping missing token usage metadata for agent CriticAgent and model mistral


CriticAgent >  APPROVED

Here's a revised version of your story:

In the secluded refuge of craggy cliffs, Elijah diligently maintained the beacon's flame. One tempestuous night, a peculiar sight captivated his gaze - a glowing map, delivered by the relentless sea.

The parchment radiated an otherworldly glow, its lines charting an unexplored course towards a remote island. Intrigued, Elijah followed the map's trail, venturing into the abyss, guided solely by his lighthouse's beam and the enigmatic allure of the glowing map.

Nights morphed into days as he navigated treacherous landscapes, but the map remained his guiding star. On the brink of the tenth day, Elijah arrived at the island, where a concealed treasure awaited him - not gold or gems, but knowledge, ancient and potent.

With newly acquired wisdom, Elijah returned to his lighthouse, the glowing map now a prized artifact. His solitude was no longer isolating, for he harbored within him secrets capable of illuminating the path 

Skipping missing token usage metadata for agent RefinerAgent and model mistral


RefinerAgent >  In the secluded refuge of craggy cliffs, Elijah diligently maintained the beacon's flame. One tempestuous night, a peculiar sight captivated his gaze - a glowing map, delivered by the relentless sea.

The parchment radiated an otherworldly glow, its lines charting an unexplored course towards a remote island. Intrigued, Elijah followed the map's trail, venturing into the abyss, guided solely by his lighthouse's beam and the enigmatic allure of the glowing map.

Nights morphed into days as he navigated treacherous landscapes, but the map remained his guiding star. On the brink of the tenth day, Elijah arrived at the island, where a concealed treasure awaited him - not gold or gems, but knowledge, ancient and potent.

With newly acquired wisdom, Elijah returned to his lighthouse, the glowing map now a prized artifact. His solitude was no longer isolating, for he harbored within him secrets capable of illuminating the path for sailors adrift at sea. From that day forth, El

Skipping missing token usage metadata for agent CriticAgent and model mistral


CriticAgent >  APPROVED

The story is well-written and engaging, with a clear narrative arc and strong character development. Here are a few suggestions to further enhance the story:

1. Consider adding more descriptive language to set the scene and immerse readers in Elijah's world. For example, you could describe the lighthouse in greater detail or provide more vivid descriptions of the stormy night when Elijah discovers the glowing map.
2. To make the story more dynamic, you could introduce a conflict or obstacle that Elijah must overcome on his journey to the island. This would add tension and suspense, making the eventual discovery of the treasure more satisfying for readers.
3. Finally, consider expanding on the theme of wisdom and guidance. You could delve deeper into how Elijah's newfound knowledge helps sailors find their way home, or explore how it affects his own life and relationships. This would give the story a deeper emotional resonance and make it more memorable for rea

Skipping missing token usage metadata for agent RefinerAgent and model mistral


RefinerAgent >  In the secluded refuge of craggy cliffs, Elijah diligently maintained the beacon's flame. The lighthouse stood tall and proud against the backdrop of the stormy sea, its powerful light piercing through the darkness. One tempestuous night, a peculiar sight captivated his gaze - a glowing map, delivered by the relentless waves.

The parchment radiated an otherworldly glow, its lines charting an unexplored course towards a remote island. Intrigued, Elijah followed the map's trail, venturing into the abyss, guided solely by his lighthouse's beam and the enigmatic allure of the glowing map.

Nights morphed into days as he navigated treacherous landscapes, but the map remained his guiding star. Along the way, Elijah faced numerous obstacles - raging storms, treacherous terrain, and even a band of pirates who sought to claim the treasure for themselves. But with determination and the wisdom imparted by the glowing map, he persevered.

On the brink of the tenth day, Elijah arri

## 6. Notes

This local version keeps the ADK workflow patterns, but it is offline-first. Live web search is intentionally removed so the notebook can run entirely against a local Mistral server.

If you want a stronger local model, set `MISTRAL_MODEL` before launching the notebook and point `OLLAMA_BASE_URL` at your server.